# Lesson 7.1 · How a tree decides

A decision tree is a **sequence of yes/no questions** about the features ("is the income above 24,000?") that splits the data into groups that are more and more homogeneous. At the end each group (a **leaf**) gives a prediction: the most frequent class, or the mean of the target.
It is the easiest model to read and to explain, and it is the building block of random forest and XGBoost (modules 8 and 9).

**What you will learn**

- How the best question is chosen at each step (Gini impurity, entropy, gain).
- How to read a tree: `export_text`, the path of an example, leaf probability.
- What `fit` does and why a tree does not need scaling.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

data = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(data.data, data.target, test_size=0.3, stratify=data.target, random_state=0)
tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_tr, y_tr)
print(export_text(tree, feature_names=list(data.feature_names)))
print("accuracy: train", round(tree.score(X_tr, y_tr), 3), "| test", round(tree.score(X_te, y_te), 3))

## How a question is chosen

At each node the algorithm tries **every feature and every possible threshold** (the midpoints between consecutive values) and picks the split that makes the two child groups **purer**. Purity is measured with the **Gini impurity**

$$G = 1 - \sum_k p_k^2$$

where $p_k$ is the fraction of class $k$ in the node: it is 0 if the node is pure and 0.5 (with two classes) if it is half and half. The **gain** of a split is the impurity of the parent minus the weighted average (by number of examples) of the impurities of the children. The alternative is
the **entropy** $-\sum_k p_k\log_2 p_k$: the two measures almost always give the same trees (exercise F1).

In [ ]:
def gini(y):
    p = np.bincount(y) / len(y)
    return 1 - (p ** 2).sum()

# the best threshold of the feature 'worst perimeter' at the root, searched by hand
x = X_tr[:, list(data.feature_names).index("worst perimeter")]
thresholds = np.unique(x)
thresholds = (thresholds[:-1] + thresholds[1:]) / 2
def gain(s):
    left, right = y_tr[x <= s], y_tr[x > s]
    return gini(y_tr) - (len(left) * gini(left) + len(right) * gini(right)) / len(y_tr)
g = np.array([gain(s) for s in thresholds])
print(f"impurity of the root: {gini(y_tr):.3f} | best threshold for 'worst perimeter': {thresholds[g.argmax()]:.2f} (gain {g.max():.3f})")
print("threshold chosen by the tree at the root:", round(tree.tree_.threshold[0], 2), "on", data.feature_names[tree.tree_.feature[0]])

The tree picked exactly `worst perimeter`, with the same threshold found by hand (106.1). After choosing, it repeats the same procedure **in every child group** (a *greedy* algorithm: it optimizes one step at a time, without looking ahead) until a stopping criterion halts it:
maximum depth, minimum number of examples in a leaf, pure node. Without limits it goes on until every leaf is pure, that is it **memorizes** the training set.

## Reading the prediction

Each leaf holds the fractions of the training classes that ended up there: `predict_proba` returns those fractions and `predict` the most frequent class. With pure leaves the probabilities are 0 and 1 (like KNN with `k = 1`, lesson 5.1).

In [ ]:
x0 = X_te[:1]
path = tree.decision_path(x0).indices                       # the nodes the example goes through
print("nodes visited:", path, "| leaf:", tree.apply(x0))
for n in path[:-1]:
    f, s = tree.tree_.feature[n], tree.tree_.threshold[n]
    print(f"  node {n}: {data.feature_names[f]} = {x0[0, f]:.2f} {'<=' if x0[0, f] <= s else '>'} {s:.2f}")
print("class proportions in the leaf:", tree.predict_proba(x0).round(3), "-> prediction", tree.predict(x0))

## Trees and scales

Every question is a threshold on **a single** feature, so a tree is not affected by the scale or by increasing monotonic transformations (logarithm, square root...): changing the units moves the thresholds but not the order of the examples. **There is no need to standardize** (exercise F2).

> **Common mistake.** Standardizing the features "for a tree" or using the log transformation to "help it": it does not change the predictions. It is wasted time (and, if done badly, a source of leakage with no benefit).

## Check your understanding

> *Click on a question to show its answer.*

<details><summary>Why is a tree a greedy algorithm?</summary>

Because at each node it chooses the best split **for that node**, without considering whether a worse split now would allow better splits later. Finding the optimal tree is an NP-hard problem, and the greedy approach gives very good trees in time linear in the depth.

</details>

<details><summary>What does it mean that a tree's `predict_proba` is a leaf fraction?</summary>

That the probability for an example is the proportion of training examples of that class in the leaf where it lands. With small or pure leaves it is very noisy (0 or 1 with few examples) and it is not calibrated.

</details>